In [0]:

from pyspark.sql import functions as F

CATALOG = "marketplace_catalog"
SCHEMA = "team6_marketplace"

def silver(name):
    return spark.table(f"{CATALOG}.{SCHEMA}.silver_{name}")

def save_delta(df, table_name):
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{CATALOG}.{SCHEMA}.{table_name}")
    )

print("Data modeling configuration completed.")


Data modeling configuration completed.


In [0]:

customers = silver("customers")

dim_customer = (
    customers
    .select(
        "customer_id",
        "customer_unique_id",
        "customer_zip_code_prefix",
        "customer_city",
        "customer_state"
    )
    .dropDuplicates(["customer_id"])
)

save_delta(dim_customer, "dim_customer")

print("dim_customer created:", dim_customer.count())


dim_customer created: 99441


In [0]:

sellers = silver("sellers")

dim_seller = (
    sellers
    .select(
        "seller_id",
        "seller_zip_code_prefix",
        "seller_city",
        "seller_state"
    )
    .dropDuplicates(["seller_id"])
)

save_delta(dim_seller, "dim_seller")

print("dim_seller created:", dim_seller.count())


dim_seller created: 3095


In [0]:

products = silver("products")

dim_product = (
    products
    .dropDuplicates(["product_id"])
)

save_delta(dim_product, "dim_product")

print("dim_product created:", dim_product.count())


dim_product created: 32951


In [0]:

orders = silver("orders")

date_source = (
    orders
    .select(
        F.to_date("order_purchase_timestamp").alias("full_date")
    )
    .filter(F.col("full_date").isNotNull())
    .distinct()
)

dim_date = (
    date_source
    .withColumn(
        "date_key",
        F.date_format("full_date", "yyyyMMdd").cast("int")
    )
    .withColumn("year", F.year("full_date"))
    .withColumn("quarter", F.quarter("full_date"))
    .withColumn("month", F.month("full_date"))
    .withColumn("month_name", F.date_format("full_date", "MMMM"))
    .withColumn("day", F.dayofmonth("full_date"))
    .withColumn("day_of_week", F.date_format("full_date", "EEEE"))
)

save_delta(dim_date, "dim_date")

print("dim_date created:", dim_date.count())


dim_date created: 634


In [0]:

items = silver("order_items")
orders = silver("orders")
reviews = silver("reviews")

# Aggregate review scores at order level
review_by_order = (
    reviews
    .groupBy("order_id")
    .agg(
        F.avg(
            F.col("review_score").cast("double")
        ).alias("review_score")
    )
)

# Select required order columns
order_details = orders.select(
    "order_id",
    "customer_id",
    "order_purchase_timestamp",
    "order_status",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
)

# Build the fact table
fact_order_items = (
    items
    .join(order_details, on="order_id", how="left")
    .join(review_by_order, on="order_id", how="left")
    .withColumn(
        "date_key",
        F.date_format(
            "order_purchase_timestamp", "yyyyMMdd"
        ).cast("int")
    )
    .select(
        "order_id",
        "order_item_id",
        "product_id",
        "seller_id",
        "customer_id",
        "date_key",
        "price",
        "freight_value",
        "order_status",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "review_score"
    )
)

save_delta(fact_order_items, "fact_order_items")

print("Fact table created:", fact_order_items.count())


Fact table created: 112650


In [0]:

model_tables = [
    "dim_customer",
    "dim_seller",
    "dim_product",
    "dim_date",
    "fact_order_items"
]

for table in model_tables:
    df = spark.table(f"{CATALOG}.{SCHEMA}.{table}")
    print(
        table,
        "| Rows:", df.count(),
        "| Columns:", len(df.columns)
    )

print("Expected order-item rows:", items.count())
print("Actual fact-table rows:", fact_order_items.count())


dim_customer | Rows: 99441 | Columns: 5
dim_seller | Rows: 3095 | Columns: 4
dim_product | Rows: 32951 | Columns: 11
dim_date | Rows: 634 | Columns: 8
fact_order_items | Rows: 112650 | Columns: 12
Expected order-item rows: 112650
Actual fact-table rows: 112650


In [0]:
display(
    spark.table(
        "marketplace_catalog.team6_marketplace.fact_order_items"
    )
)

order_id,order_item_id,product_id,seller_id,customer_id,date_key,price,freight_value,order_status,order_delivered_customer_date,order_estimated_delivery_date,review_score
0010b2e5201cc5f1ae7e9c6cc8f5bd00,1,5a419dbf24a8c9718fe522b81c69f61a,3504c0cb71d7fa48d967e0e4c94d59d9,57ef317d4818cb42680fc9dfd13867ce,20170911,48.9,16.6,delivered,2017-09-23T13:21:21.000Z,2017-09-27T00:00:00.000Z,5.0
0011d82c4b53e22e84023405fb467e57,1,c389f712c4b4510bc997cee93e8b1a28,bfd27a966d91cfaafdb25d076585f0da,2013d892495e1a101d742d533d2d1119,20180116,289.0,26.33,delivered,2018-01-26T22:14:02.000Z,2018-02-15T00:00:00.000Z,4.0
00143d0f86d6fbd9f9b38ab440ac16f5,2,e95ee6822b66ac6058e2e4aff656071a,a17f621c590ea0fab3d5d883e1630ec6,2e45292979b9b2700ea09560eeb0f803,20171016,21.33,15.1,delivered,2017-10-27T18:43:56.000Z,2017-11-06T00:00:00.000Z,null
0014ae671de39511f7575066200733b7,1,23365beed316535b4105bd800c46670e,92eb0f42c21942b6552362b9b114707d,41065d9dcea52218c3943d2eed072b97,20170522,16.5,14.1,delivered,2017-06-07T13:52:52.000Z,2017-06-13T00:00:00.000Z,5.0
0016dfedd97fc2950e388d2971d718c7,1,4089861a1bd4685da70bddd6b4f974f1,a35124e2d763d7ca3fbe3b97d143200f,2c8b917c5d7dd720ebe36a5ed3b501ec,20170428,49.75,20.8,delivered,2017-05-22T18:32:21.000Z,2017-06-01T00:00:00.000Z,5.0
001ab0a7578dd66cd4b0a71f5b6e1e41,2,0b0172eb0fd18479d29c3bc122c058c2,5656537e588803a555b8eb41f07a944b,8bb3bef4e75a95524235cdc11a7331af,20171227,24.89,17.63,delivered,2018-01-17T20:43:29.000Z,2018-01-29T00:00:00.000Z,4.0
002175704e8b209f61b9ad5cfd92b60e,1,e6b6e13cf71449a457269f425b89dc74,b2ba3715d723d245138f291a6fe42594,a562db3c7cb9a68947debd30879b491e,20180422,109.9,13.21,delivered,2018-05-02T20:38:44.000Z,2018-05-14T00:00:00.000Z,4.0
002d040018d12a3853c059f7f23ab5b1,1,97c58626cc3e15d514d2095f69650220,fe2032dab1a61af8794248c8196565c9,d7a77284d0a56f6b1ecb8c344b60d3e4,20171119,155.0,14.84,delivered,2017-11-27T18:54:09.000Z,2017-12-08T00:00:00.000Z,3.0
003423b755b562962a6225a8de40d12e,1,84904413b82eee333a3f79137e8d197e,23c38debaffe4a25a30fdbd9b586a13f,18f1bb6325d50619d5c13b7a25b869fd,20180708,232.75,28.58,delivered,2018-07-24T21:33:38.000Z,2018-08-08T00:00:00.000Z,3.0
0035246a40f520710769010f752e7507,1,8a6187b2665118d5095f99a25fd7ba7a,4a3ca9315b744ce9f8e9374361493884,4bcf2ad7ccc061c13ff360aae5b7abcd,20170817,87.0,12.11,delivered,2017-08-28T19:14:03.000Z,2017-09-06T00:00:00.000Z,5.0
